# Working

**Request:** [T06] Did the new checkout actually win?

> New email from Sai Suram <sai@agents.agentstore.it.com>
> Subject: [T06] Did the new checkout actually win?
> Thread ID: AAQkADI1N2Y5MTE3LTE1MDctNGY0Yy1iYzQ5LWEzNmE5NzAyYzk4NQAQAEb2fqe6qodLr1uqwZDfqUc=
> 
> #

Each cell below is one run in the sandbox, in the order it happened, with whatever it printed and produced.


In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

data = {
    'Variant': ['Control', 'Variant'],
    'Visitors': [8432, 8391],
    'Conversions': [421, 502]
}
df = pd.DataFrame(data)

# Calculate Conversion Rate
df['Conversion_Rate'] = df['Conversions'] / df['Visitors']

# Calculate Relative Lift
control_conversion_rate = df[df['Variant'] == 'Control']['Conversion_Rate'].iloc[0]
variant_conversion_rate = df[df['Variant'] == 'Variant']['Conversion_Rate'].iloc[0]

relative_lift = ((variant_conversion_rate - control_conversion_rate) / control_conversion_rate) * 100

# Statistical Significance (Z-test for proportions)
# Null Hypothesis (H0): There is no difference in conversion rates between Control and Variant.
# Alternative Hypothesis (H1): There is a difference in conversion rates between Control and Variant.

count = np.array([df[df['Variant'] == 'Variant']['Conversions'].iloc[0], df[df['Variant'] == 'Control']['Conversions'].iloc[0]])
nobs = np.array([df[df['Variant'] == 'Variant']['Visitors'].iloc[0], df[df['Variant'] == 'Control']['Visitors'].iloc[0]])

stat, pval = proportions_ztest(count, nobs, alternative='two-sided')

alpha = 0.05
is_significant = pval < alpha

# Prepare output for Excel
results = df.copy()
results.loc[len(results)] = ['Relative Lift', np.nan, np.nan, relative_lift / 100] # Store as decimal for easier formatting in Excel
results.loc[len(results)] = ['Z-test Statistic', np.nan, np.nan, stat]
results.loc[len(results)] = ['P-value', np.nan, np.nan, pval]
results.loc[len(results)] = ['Significant (alpha=0.05)', np.nan, np.nan, is_significant]

# Save to Excel
output_filename = '/tmp/output/checkout_test_results.xlsx'
writer = pd.ExcelWriter(output_filename, engine='xlsxwriter')

# Write the main results to 'Analysis' sheet
df.to_excel(writer, sheet_name='Analysis', index=False)
workbook = writer.book
worksheet = writer.sheets['Analysis']

# Add headers and calculations to 'Summary' sheet
summary_df = pd.DataFrame({
    'Metric': ['Control Conversion Rate', 'Variant Conversion Rate', 'Relative Lift', 'Z-test Statistic', 'P-value', 'Statistically Significant (alpha=0.05)'],
    'Value': [
        control_conversion_rate,
        variant_conversion_rate,
        relative_lift,
        stat,
        pval,
        'Yes' if is_significant else 'No'
    ]
})
summary_df.to_excel(writer, sheet_name='Summary', index=False)
summary_worksheet = writer.sheets['Summary']

# Format percentages and other values
percent_format = workbook.add_format({'num_format': '0.00%'}) # for 2 decimal places
worksheet.set_column('C:C', None, percent_format) # Assuming 'Conversion_Rate' is in column C (0-indexed 2)

# Apply conditional formatting for 'Significant' in Summary sheet
if is_significant:
    summary_worksheet.write('B7', 'Yes', workbook.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100'})) # Green for Yes
else:
    summary_worksheet.write('B7', 'No', workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006'})) # Red for No

writer.close()

print(f'File saved to {output_filename}')


Traceback (most recent call last):
  File "<string>", line 8, in <module>
ModuleNotFoundError: No module named 'statsmodels'



[this step exited with code 1 — it did not finish]


In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import openpyxl

data = {
    'Variant': ['Control', 'Variant'],
    'Visitors': [8432, 8391],
    'Conversions': [421, 502]
}
df = pd.DataFrame(data)

# Calculate Conversion Rates
df['Conversion Rate'] = df['Conversions'] / df['Visitors']

control_conversions = df.loc[df['Variant'] == 'Control', 'Conversions'].iloc[0]
control_visitors = df.loc[df['Variant'] == 'Control', 'Visitors'].iloc[0]
variant_conversions = df.loc[df['Variant'] == 'Variant', 'Conversions'].iloc[0]
variant_visitors = df.loc[df['Variant'] == 'Variant', 'Visitors'].iloc[0]

control_conversion_rate = control_conversions / control_visitors
variant_conversion_rate = variant_conversions / variant_visitors

# Calculate Relative Lift
relative_lift = ((variant_conversion_rate - control_conversion_rate) / control_conversion_rate) * 100

# Perform Z-test for proportions
# Pooled proportion
p_pooled = (control_conversions + variant_conversions) / (control_visitors + variant_visitors)

# Standard error
SE = np.sqrt(p_pooled * (1 - p_pooled) * (1/control_visitors + 1/variant_visitors))

# Z-score
z_score = (variant_conversion_rate - control_conversion_rate) / SE

# P-value (two-tailed)
p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))

# Significance level
significance_level = 0.05
is_significant = p_value < significance_level

results = {
    'Control Conversion Rate': control_conversion_rate,
    'Variant Conversion Rate': variant_conversion_rate,
    'Relative Lift': relative_lift,
    'Z-score': z_score,
    'P-value': p_value,
    'Is Statistically Significant (at 5% level)': is_significant,
    'Test Used': 'Two-sample Z-test for proportions'
}

# Prepare data for Excel workbook
output_df = df.copy()
output_df['Conversion Rate (%)'] = output_df['Conversion Rate'] * 100

# Add summary results to a new dataframe for the workbook
summary_data = {
    'Metric': [
        'Control Conversion Rate',
        'Variant Conversion Rate',
        'Relative Lift (%)',
        'Z-score',
        'P-value',
        'Statistically Significant (5% level)',
        'Statistical Test Used'
    ],
    'Value': [
        f'{results["Control Conversion Rate"]:,.4f}',
        f'{results["Variant Conversion Rate"]:,.4f}',
        f'{results["Relative Lift"]:,.2f}',
        f'{results["Z-score"]:,.3f}',
        f'{results["P-value"]:,.4f}',
        str(results['Is Statistically Significant (at 5% level)']),
        results['Test Used']
    ]
}
summary_df = pd.DataFrame(summary_data)

# Create an Excel writer object
output_file_path = '/tmp/output/checkout_test_results.xlsx'
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    output_df.to_excel(writer, sheet_name='Raw Data & Rates', index=False)
    summary_df.to_excel(writer, sheet_name='Summary', index=False)

print(f"Conversion Rate - Control: {control_conversion_rate:.4f}")
print(f"Conversion Rate - Variant: {variant_conversion_rate:.4f}")
print(f"Relative Lift: {relative_lift:.2f}%")
print(f"Statistical Significance (5% level): {is_significant}")
print(f"Statistical Test Used: Two-sample Z-test for proportions")
print(f"P-value: {p_value:.4f}")
print(f"Z-score: {z_score:.3f}")


Traceback (most recent call last):
  File "<string>", line 8, in <module>
ModuleNotFoundError: No module named 'scipy'



[this step exited with code 1 — it did not finish]


In [3]:
import math
import pandas as pd

# Data provided
control_visitors = 8432
control_conversions = 421
variant_visitors = 8391
variant_conversions = 502

# Calculate conversion rates
control_cr = control_conversions / control_visitors
variant_cr = variant_conversions / variant_visitors

# Calculate relative lift
relative_lift = (variant_cr - control_cr) / control_cr * 100

# Perform Z-test for proportions
# Pooled proportion
p_pooled = (control_conversions + variant_conversions) / (control_visitors + variant_visitors)

# Standard error
se = math.sqrt(p_pooled * (1 - p_pooled) * (1/control_visitors + 1/variant_visitors))

# Z-score
z_score = (variant_cr - control_cr) / se

# For a 5% significance level (alpha = 0.05) in a two-tailed test, the critical Z-value is approximately 1.96.
# If abs(z_score) > 1.96, the difference is statistically significant.

is_significant = abs(z_score) > 1.96

# Prepare results for the Excel workbook
results_data = {
    'Metric': [
        'Control Visitors',
        'Control Conversions',
        'Control Conversion Rate',
        'Variant Visitors',
        'Variant Conversions',
        'Variant Conversion Rate',
        'Relative Lift (%)',
        'Z-score',
        'Statistically Significant (5% level)'
    ],
    'Value': [
        control_visitors,
        control_conversions,
        control_cr,
        variant_visitors,
        variant_conversions,
        variant_cr,
        relative_lift,
        z_score,
        is_significant
    ]
}

results_df = pd.DataFrame(results_data)

# Save to an Excel file
output_file = '/tmp/output/checkout_test_results.xlsx'
writer = pd.ExcelWriter(output_file, engine='xlsxwriter')
results_df.to_excel(writer, sheet_name='Analysis Results', index=False)

# Add a summary sheet
summary_df = pd.DataFrame({
    'Key Finding': [
        'Conversion Rate Comparison',
        'Relative Lift',
        'Statistical Significance'
    ],
    'Detail': [
        f'Control: {control_cr:.2%}, Variant: {variant_cr:.2%}',
        f'{relative_lift:.2f}%',
        'Yes' if is_significant else 'No'
    ]
})
summary_df.to_excel(writer, sheet_name='Summary', index=False)

writer.close()

print(f"Control Conversion Rate: {control_cr:.4f}")
print(f"Variant Conversion Rate: {variant_cr:.4f}")
print(f"Relative Lift: {relative_lift:.2f}%")
print(f"Z-score: {z_score:.2f}")
print(f"Statistically Significant (5% level): {is_significant}")


Control Conversion Rate: 0.0499
Variant Conversion Rate: 0.0598
Relative Lift: 19.82%
Z-score: 2.82
Statistically Significant (5% level): True



[files written: checkout_test_results.xlsx]
